In [32]:
def Threshold_Measure_Calcium_Imaging(input_folder):
    """
    Processes a folder of TIFF images by grouping them based on everything before the "C_#" part of their filenames.
    The threshold is calculated from the last image in each group and applied to all images in that group.
    Results are saved numerically in an Excel file, with each group on a separate sheet.
    Additionally, trace plots of white area percentages are generated and saved to a "0_Plots" folder.
    Compiled trace plots are created for image sets that share the same base name
    (everything before the final _NN, e.g. Co_0w3_15Glu_01/02/03 -> base Co_0w3_15Glu).
    """
    import os
    import cv2
    from collections import defaultdict
    import re
    import pandas as pd
    from natsort import natsorted
    import matplotlib.pyplot as plt

    # Create output folders if they don't exist
    output_folder = os.path.join(input_folder, "0_Thresholded")
    plots_folder = os.path.join(input_folder, "0_Plots")
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(plots_folder, exist_ok=True)

    # Group images by everything before "_C_#"
    grouped_files = defaultdict(list)
    for file_name in os.listdir(input_folder):
        if file_name.lower().endswith(('.tiff', '.tif')):
            match = re.match(r"^(.*)_C_\d+", file_name)
            if match:
                group_key = match.group(1)  # e.g. Co_0w3_15Glu_01
                grouped_files[group_key].append(file_name)

    if not grouped_files:
        print("No TIFF files found in the folder.")
        return

    grouped_results = {}
    compiled_results = defaultdict(list)  # base_name -> list[(group_key, [trace])]

    # Process each group
    for group_key, file_names in grouped_files.items():
        print(f"\nProcessing group: {group_key}")

        file_names = natsorted(file_names)

        last_image_path = os.path.join(input_folder, file_names[-1])
        last_image = cv2.imread(last_image_path, cv2.IMREAD_GRAYSCALE)
        if last_image is None:
            print(f"Error: Unable to read the last image in group {group_key}: {file_names[-1]}")
            continue

        otsu_threshold, _ = cv2.threshold(
            last_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        print(
            f"Calculated Otsu's threshold for group {group_key} from the last image: "
            f"{file_names[-1]} (Threshold: {otsu_threshold})"
        )

        group_results = []

        for file_name in file_names:
            input_path = os.path.join(input_folder, file_name)
            image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                print(f"Error: Unable to read image: {file_name}")
                continue

            _, thresholded = cv2.threshold(
                image, otsu_threshold, 255, cv2.THRESH_BINARY
            )

            output_path = os.path.join(output_folder, file_name)
            cv2.imwrite(output_path, thresholded)
            print(f"Processed and saved: {output_path}")

            total_pixels = image.size
            white_pixels = cv2.countNonZero(thresholded)
            white_area_percentage = (white_pixels / total_pixels) * 100

            group_results.append(
                {
                    "Image Name": file_name,
                    "White Area (%)": white_area_percentage,
                }
            )

        # store full trace once per group for compiled plots
        df_group = pd.DataFrame(group_results)
        grouped_results[group_key] = df_group

        # ---- compute base name for compiled plots: strip last _NN or _NN_ from group_key ----
        # accepts "Co_0w3_15Glu_01", "Co_0w3_15Glu_01_", "Tri_Ctrl_03", "Tri_Ctrl_03_"
        base_name_match = re.match(r"^(.*)_\d+_?$", group_key)
        if base_name_match:
            base_name = base_name_match.group(1)
            compiled_results[base_name].append(
                (group_key, df_group["White Area (%)"].tolist())
            )

        # -------- individual trace plot (NO legend) --------
        plt.figure(figsize=(10, 6))
        plt.plot(
            range(len(df_group)),
            df_group["White Area (%)"],
            marker="o",
        )
        plt.title(f"White Area Trace Plot: {group_key}")
        plt.xlabel("Image Index")
        plt.ylabel("White Area (%)")
        plt.grid(True)
        plot_path = os.path.join(plots_folder, f"{group_key}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved trace plot for group {group_key}: {plot_path}")

    # -------- compiled trace plots (keep legend, for ANY number of traces >= 1) --------
    for base_name, traces in compiled_results.items():
        if not traces:
            continue
        plt.figure(figsize=(10, 6))
        for group_key, trace_vals in traces:
            plt.plot(
                range(len(trace_vals)),
                trace_vals,
                marker="o",
                label=group_key,
            )
        plt.title(f"Compiled White Area Trace Plot: {base_name}")
        plt.xlabel("Image Index")
        plt.ylabel("White Area (%)")
        plt.grid(True)
        plt.legend()
        compiled_plot_path = os.path.join(
            plots_folder, f"{base_name}_compiled.png"
        )
        plt.savefig(compiled_plot_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(
            f"Saved compiled trace plot for base name {base_name}: "
            f"{compiled_plot_path}"
        )

    # -------- Excel output --------
    excel_output_path = os.path.join(output_folder, "0_Threshold_Results.xlsx")
    with pd.ExcelWriter(excel_output_path) as writer:
        for group_key, df in grouped_results.items():
            df.to_excel(writer, sheet_name=group_key[:31], index=False)

    print(f"\nResults saved to: {excel_output_path}")

Threshold_Measure_Calcium_Imaging('/Users/allisonpickle/Desktop/Calcium Imaging')


Processing group: Co_Ctrl_05
Calculated Otsu's threshold for group Co_Ctrl_05 from the last image: Co_Ctrl_05_C_89.tiff (Threshold: 22.0)
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_0.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_1.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_2.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_3.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_4.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_5.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_6.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Imaging/0_Thresholded/Co_Ctrl_05_C_7.tiff
Processed and saved: /Users/allisonpickle/Desktop/Calcium Ima

In [5]:
def Threshold_Measure_Calcium_Imaging_Normalized(input_folder):
    import os
    import cv2
    from collections import defaultdict
    import re
    import pandas as pd
    from natsort import natsorted
    import matplotlib.pyplot as plt

    # --- Setup Folders ---
    output_folder = os.path.join(input_folder, "0_Thresholded")
    plots_folder = os.path.join(input_folder, "0_Plots")
    norm_root = os.path.join(input_folder, "0_Normalized_Output")
    norm_plots_folder = os.path.join(norm_root, "0_Normalized_Plots")

    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(plots_folder, exist_ok=True)
    os.makedirs(norm_root, exist_ok=True)
    os.makedirs(norm_plots_folder, exist_ok=True)

    # --- Group Files ---
    grouped_files = defaultdict(list)
    for file_name in os.listdir(input_folder):
        if file_name.lower().endswith((".tiff", ".tif")):
            m = re.match(r"^(.*)_C_\d+", file_name)
            if m:
                group_key = m.group(1)
                grouped_files[group_key].append(file_name)

    if not grouped_files:
        print("No TIFF files found in the folder.")
        return

    grouped_results = {}
    grouped_results_norm = {}
    
    # Dictionaries to store traces for compiled plots
    compiled_results = defaultdict(list)
    compiled_results_norm = defaultdict(list)

    # --- Process Images (Alphabetically) ---
    sorted_group_keys = natsorted(grouped_files.keys())
    
    for group_key in sorted_group_keys:
        file_names = grouped_files[group_key]
        file_names = natsorted(file_names)

        last_image_path = os.path.join(input_folder, file_names[-1])
        last_image = cv2.imread(last_image_path, cv2.IMREAD_GRAYSCALE)
        if last_image is None:
            continue

        otsu_threshold, _ = cv2.threshold(last_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        group_results = []
        for file_name in file_names:
            input_path = os.path.join(input_folder, file_name)
            image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue

            _, thresholded = cv2.threshold(image, otsu_threshold, 255, cv2.THRESH_BINARY)
            output_path = os.path.join(output_folder, file_name)
            cv2.imwrite(output_path, thresholded)

            total_pixels = image.size
            white_pixels = cv2.countNonZero(thresholded)
            white_area_percentage = (white_pixels / total_pixels) * 100
            group_results.append({"Image Name": file_name, "White Area (%)": white_area_percentage})

        df_group = pd.DataFrame(group_results)
        grouped_results[group_key] = df_group

        if not df_group.empty:
            first_val = df_group["White Area (%)"].iloc[0]
            norm_values = [0.0] * len(df_group) if first_val == 0 else (df_group["White Area (%)"] / first_val).tolist()
        else:
            norm_values = []

        df_group_norm = df_group.copy()
        df_group_norm["White Area (Normalized)"] = norm_values
        grouped_results_norm[group_key] = df_group_norm

        # --- Base Name Extraction for Compiled Plots ---
        parts = group_key.split("_")
        if len(parts) > 1 and parts[-1].isdigit():
            base_name = "_".join(parts[:-1])
        else:
            base_name = group_key

        # Store for compiled plots
        if not df_group.empty:
            compiled_results[base_name].append((group_key, df_group["White Area (%)"].tolist()))
            compiled_results_norm[base_name].append((group_key, norm_values))

        # --- INDIVIDUAL PLOTS (Raw & Normalized) ---
        if not df_group.empty:
            # Raw individual
            plt.figure(figsize=(10, 6))
            plt.plot(range(len(df_group)), df_group["White Area (%)"], marker="o")
            plt.title(f"White Area Trace Plot: {group_key}")
            plt.xlabel("Image Index")
            plt.ylabel("White Area (%)")
            plt.grid(True)
            saved_raw_plot = os.path.join(plots_folder, f"{group_key}.png")
            plt.savefig(saved_raw_plot, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved Individual: {saved_raw_plot}")

            # Normalized individual
            plt.figure(figsize=(10, 6))
            plt.plot(range(len(norm_values)), norm_values, marker="o")
            plt.title(f"Normalized White Area Trace Plot: {group_key}")
            plt.xlabel("Image Index")
            plt.ylabel("White Area (Normalized to first point)")
            plt.grid(True)
            saved_norm_plot = os.path.join(norm_plots_folder, f"{group_key}.png")
            plt.savefig(saved_norm_plot, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved Normalized: {saved_norm_plot}")

    print("\nCreating COMPILED plots...")
    
    # --- COMPILED RAW PLOTS ---
    for base_name, traces in compiled_results.items():
        traces = sorted(traces, key=lambda x: x[0])
        plt.figure(figsize=(14, 8))
        for group_key, trace_vals in traces:
            plt.plot(range(len(trace_vals)), trace_vals, marker="o", linewidth=2, markersize=5, label=group_key)
        plt.title(f"Compiled: {base_name}")
        plt.xlabel("Image Index")
        plt.ylabel("White Area (%)")
        plt.grid(True, alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
        plt.tight_layout()
        save_path = os.path.join(plots_folder, f"{base_name}_compiled.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved Compiled Raw: {save_path} ({len(traces)} traces)")

    # --- COMPILED NORMALIZED PLOTS ---
    for base_name, traces in compiled_results_norm.items():
        traces = sorted(traces, key=lambda x: x[0])
        plt.figure(figsize=(14, 8))
        for group_key, norm_trace in traces:
            plt.plot(range(len(norm_trace)), norm_trace, marker="o", linewidth=2, markersize=5, label=group_key)
        plt.title(f"Compiled Normalized: {base_name}")
        plt.xlabel("Image Index")
        plt.ylabel("White Area (Normalized)")
        plt.grid(True, alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        save_path = os.path.join(norm_plots_folder, f"{base_name}_compiled.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved Compiled Normalized: {save_path} ({len(traces)} traces)")

    # --- EXCEL EXPORT (Alphabetical Sheets) ---
    print("\nExporting Excel files...")
    with pd.ExcelWriter(os.path.join(output_folder, "0_Threshold_Results.xlsx")) as writer:
        for group_key in natsorted(grouped_results.keys()):
            df = grouped_results[group_key]
            df.to_excel(writer, sheet_name=group_key[:31], index=False)

    with pd.ExcelWriter(os.path.join(norm_root, "0_Normalized_Results.xlsx")) as writer:
        for group_key in natsorted(grouped_results_norm.keys()):
            df = grouped_results_norm[group_key]
            df.to_excel(writer, sheet_name=group_key[:31], index=False)

    print("✓ Done!")

Threshold_Measure_Calcium_Imaging_Normalized('/Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis')

Saved Individual: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Plots/Co_0w3_15Glu_01.png
Saved Normalized: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Normalized_Output/0_Normalized_Plots/Co_0w3_15Glu_01.png
Saved Individual: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Plots/Co_0w3_15Glu_02.png
Saved Normalized: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Normalized_Output/0_Normalized_Plots/Co_0w3_15Glu_02.png
Saved Individual: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Plots/Co_0w3_15Glu_03.png
Saved Normalized: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Normalized_Output/0_Normalized_Plots/Co_0w3_15Glu_03.png
Saved Individual: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Plots/Co_0w3_25Glu_01.png
Saved Normalized: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Normalized_Output/0_Normalized_Plots/Co_0w3_25Glu_01.png
Saved Individual: /U